# Relación muchos a muchos 

**Resumen**: Usando una biblioteca iTunes en un archivo CSV (library.csv), se produjeron tablas correctamente normalizadas según lo especificado. Las tablas elaboradas son _track_, _album_, _artist_, y _tracktoartist_. En la tabla track se pusieron los datos del archivo csv.

# Musical Track Database plus Artists 

## Librerías y definiciones

In [30]:
# Importar librerias 
import os
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
from tabulate import tabulate

In [31]:
# definición para CREAT, aLTER TABLE  
def ejecutar_sql(sql):
    try:
        cur.execute(sql)
        conn.commit()
        print("Operación realizada correctamente")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [32]:
#definición para SELECT 
from tabulate import tabulate

def mostrar_tabla(sql):
    try:
        cur.execute(sql)

        filas = cur.fetchall()
        columnas = [desc[0] for desc in cur.description]

        print(tabulate(
            filas,
            headers=columnas,
            tablefmt="psql"
        ))

    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [33]:
# definición para ver tablas usando pandas
def consultar_df(sql):
    try:
        df = pd.read_sql(sql, conn)
        return df

    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [34]:
# definición para insertar datos 
def insertar_datos(sql, datos=None):
    try:
        if datos is None: #dejamos como condición los datos
            cur.execute(sql)
        else:
            cur.execute(sql, datos)
            
        conn.commit()
        print("Datos insertados correctamente")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [35]:
# definición para UPDATE 
def actualizar_datos(sql, datos=None):
    try:
        if datos is None:
            cur.execute(sql) 
        else:
            cur.execute(sql, datos)
        conn.commit()
        print("Datos actualizados")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

## Conexión con PostgreSQL

In [37]:
# datos 
host = "pg.pg4e.com"
port = 5432
database = "pg4e_619b3a797b"
user = "pg4e_619b3a797b"
password = "******************"

#estableciendo conexión 
conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=user,
    password=password
)
cur = conn.cursor()
print("Conexión exitosa.")

Conexión exitosa.


## Actividad

Creación, altearación, actualización y muestra de tablas

In [40]:
# reescribiendo tablas 
ejecutar_sql('''
DROP TABLE IF EXISTS album CASCADE;
CREATE TABLE album (
    id SERIAL,
    title VARCHAR(128) UNIQUE,
    PRIMARY KEY(id)
);

DROP TABLE IF EXISTS track CASCADE;
CREATE TABLE track (
    id SERIAL,
    title TEXT, 
    artist TEXT, 
    album TEXT, 
    album_id INTEGER REFERENCES album(id) ON DELETE CASCADE,
    count INTEGER, 
    rating INTEGER, 
    len INTEGER,
    PRIMARY KEY(id)
);

DROP TABLE IF EXISTS artist CASCADE;
CREATE TABLE artist (
    id SERIAL,
    name VARCHAR(128) UNIQUE,
    PRIMARY KEY(id)
);

DROP TABLE IF EXISTS tracktoartist CASCADE;
CREATE TABLE tracktoartist (
    id SERIAL,
    track VARCHAR(128),
    track_id INTEGER REFERENCES track(id) ON DELETE CASCADE,
    artist VARCHAR(128),
    artist_id INTEGER REFERENCES artist(id) ON DELETE CASCADE,
    PRIMARY KEY(id)
);
''')

Operación realizada correctamente


Copiar datos del csv a la tabla track

In [42]:
# Usar \copy para poner los datos en la tabla

with open("C:/Users/black/Coursera/PostgreSQL/library.csv", "r", encoding="utf-8") as archivo:
    cur.copy_expert(
        """
        COPY track(
            title,
            artist,
            album,
            count,
            rating,
            len
        )
        FROM STDIN
        WITH CSV DELIMITER ',';
        """,
        archivo
    )

conn.commit()

In [43]:
mostrar_tabla('''
SELECT * FROM track LIMIT 5;
''')

+------+----------------------------+--------------+-------------------+------------+---------+----------+-------+
|   id | title                      | artist       | album             | album_id   |   count |   rating |   len |
|------+----------------------------+--------------+-------------------+------------+---------+----------+-------|
|    1 | Another One Bites The Dust | Queen        | Greatest Hits     |            |      55 |      100 |   217 |
|    2 | Asche Zu Asche             | Rammstein    | Herzeleid         |            |      79 |      100 |   231 |
|    3 | Beauty School Dropout      | Various      | Grease            |            |      48 |      100 |   239 |
|    4 | Black Dog                  | Led Zeppelin | IV                |            |     109 |      100 |   296 |
|    5 | Bring The Boys Back Home   | Pink Floyd   | The Wall [Disc 2] |            |      33 |      100 |    87 |
+------+----------------------------+--------------+-------------------+--------

Insertar title de track a album

In [45]:
insertar_datos('''
INSERT INTO album (title) SELECT DISTINCT album FROM track;
''')

Datos insertados correctamente


Actualizar 

In [47]:
actualizar_datos('''
UPDATE track SET album_id = (SELECT album.id FROM album WHERE album.title = track.album);
''')

Datos actualizados


Insertar track, artist de track a tracktoartist

In [49]:
insertar_datos('''
INSERT INTO tracktoartist (track, artist) SELECT DISTINCT title,artist FROM track;
''')

Datos insertados correctamente


Insertar más datos

In [51]:
insertar_datos('''
INSERT INTO artist (name) SELECT DISTINCT artist FROM track;
''')

Datos insertados correctamente


Actualizar ID's

In [53]:
actualizar_datos('''
UPDATE tracktoartist SET track_id = (SELECT track.id FROM track WHERE track.title = tracktoartist.track);
''')

Datos actualizados


In [54]:
actualizar_datos('''
UPDATE tracktoartist SET artist_id = (SELECT artist.id FROM artist WHERE artist.name = tracktoartist.artist);
''')

Datos actualizados


In [71]:
ejecutar_sql('''
-- We are now done with these text fields
ALTER TABLE track DROP COLUMN album;
ALTER TABLE track DROP COLUMN artist;
ALTER TABLE tracktoartist DROP COLUMN track;
ALTER TABLE tracktoartist DROP COLUMN artist;
''')

Operación realizada correctamente


## Parte final: AUTOGRADE

In [73]:
mostrar_tabla('''
SELECT track.title, album.title, artist.name
FROM track
JOIN album ON track.album_id = album.id
JOIN tracktoartist ON track.id = tracktoartist.track_id
JOIN artist ON tracktoartist.artist_id = artist.id
ORDER BY track.title
LIMIT 3;
''')

+----------------------------+------------------------------------+-----------------------+
| title                      | title                              | name                  |
|----------------------------+------------------------------------+-----------------------|
| A Boy Named Sue (live)     | The Legend Of Johnny Cash          | Johnny Cash           |
| A Brief History of Packets | Computing Conversations            | IEEE Computer Society |
| Aguas De Marco             | Natural Wonders Music Sampler 1999 | Rosa Passos           |
+----------------------------+------------------------------------+-----------------------+
